## FAISS

Facebook AI Similarity Search(FAISS) is library for efficient similarity search and clustering of dense vectors. It contains algorithms that search in sets of vectors of any size, up to ones that possibly do not fit in RAM . It also contains supporting code for evaluation and parameter tuning

In [43]:
from langchain_community.document_loaders import TextLoader,PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter,RecursiveCharacterTextSplitter



# loader = TextLoader("speech.txt")
loader = PyPDFLoader("Abhishek Jaiswar - Web Developer.pdf")
documents = loader.load()
# text_splitter = CharacterTextSplitter(chunk_size=500,chunk_overlap=30)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=30)
docs = text_splitter.split_documents(documents)


In [44]:
docs

[Document(metadata={'producer': 'pdfTeX-1.40.24', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-01-18T15:53:18+00:00', 'author': '', 'keywords': '', 'moddate': '2026-01-18T15:53:18+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.24 (TeX Live 2022) kpathsea version 6.3.4', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'Abhishek Jaiswar - Web Developer.pdf', 'total_pages': 1, 'page': 0, 'page_label': ''}, page_content='9082491824\nThane, Maharashtra\naj958601@gmail.com\nAbhishek Jaiswar\nWeb Developer\nPortfolio: abhishek-js.netlify.app\ngithub.com/ abhishek-jaiswar\nlinkedin.com/in/ abhishek-jaiswar\nPassionate Web Developer with 1 year of professional experience at TCS and 3 years of hands-on personal project development.\nSkilled in React, Tailwind CSS, Redux, Node.js, Express.js, and REST APIs.\nEDUCATION\nB.E. Computer Engineering, Terna Engineering College 2020-2024\n• Average CGPA : 8.22'),
 Document(metadata={'producer': 'pdfTeX-1.4

In [59]:
embeddings =  OllamaEmbeddings(
    model="qwen3-embedding:0.6b"
)
# Save to FAISS vector store
# db = FAISS.from_documents(docs,embeddings)
# db

# Save locally

db = FAISS.from_documents(docs, embeddings)
db.save_local("faiss_index")

# Load from local directory
db = FAISS.load_local("faiss_index", embeddings,allow_dangerous_deserialization=True)
db


In [60]:
## querying

query = "Abhishek Skills ?"
docs = db.similarity_search(query)
docs[0].page_content

'9082491824\nThane, Maharashtra\naj958601@gmail.com\nAbhishek Jaiswar\nWeb Developer\nPortfolio: abhishek-js.netlify.app\ngithub.com/ abhishek-jaiswar\nlinkedin.com/in/ abhishek-jaiswar\nPassionate Web Developer with 1 year of professional experience at TCS and 3 years of hands-on personal project development.\nSkilled in React, Tailwind CSS, Redux, Node.js, Express.js, and REST APIs.\nEDUCATION\nB.E. Computer Engineering, Terna Engineering College 2020-2024\n• Average CGPA : 8.22'

In [61]:
query="Abhishek Projects"
docs = db.similarity_search(query)
docs[0].page_content

'9082491824\nThane, Maharashtra\naj958601@gmail.com\nAbhishek Jaiswar\nWeb Developer\nPortfolio: abhishek-js.netlify.app\ngithub.com/ abhishek-jaiswar\nlinkedin.com/in/ abhishek-jaiswar\nPassionate Web Developer with 1 year of professional experience at TCS and 3 years of hands-on personal project development.\nSkilled in React, Tailwind CSS, Redux, Node.js, Express.js, and REST APIs.\nEDUCATION\nB.E. Computer Engineering, Terna Engineering College 2020-2024\n• Average CGPA : 8.22'

In [ ]:
## Retriever
# We can also convert the vectorstore into a Retriever class. This allows us to easily use it in other LangChain methods, which largely work with retrievers


In [62]:
retriver = db.as_retriever()
retriver.invoke(query)

[Document(id='c9c91f63-62e8-4771-95cb-51c3226c8651', metadata={'producer': 'pdfTeX-1.40.24', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-01-18T15:53:18+00:00', 'author': '', 'keywords': '', 'moddate': '2026-01-18T15:53:18+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.24 (TeX Live 2022) kpathsea version 6.3.4', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'Abhishek Jaiswar - Web Developer.pdf', 'total_pages': 1, 'page': 0, 'page_label': ''}, page_content='9082491824\nThane, Maharashtra\naj958601@gmail.com\nAbhishek Jaiswar\nWeb Developer\nPortfolio: abhishek-js.netlify.app\ngithub.com/ abhishek-jaiswar\nlinkedin.com/in/ abhishek-jaiswar\nPassionate Web Developer with 1 year of professional experience at TCS and 3 years of hands-on personal project development.\nSkilled in React, Tailwind CSS, Redux, Node.js, Express.js, and REST APIs.\nEDUCATION\nB.E. Computer Engineering, Terna Engineering College 2020-2024\n• Average CGPA : 8.22'),


In [63]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.llms import Ollama

# 5. LLM + prompt for generation
llm = Ollama(model="gemma3:1b")

prompt = ChatPromptTemplate.from_template(
    """Answer the question based ONLY on the following context:
{context}

Question: {question}"""
)
chain = (
    {"context": retriver, "question": lambda x: x}  # RunnablePassthrough for question
    | prompt
    | llm
    | StrOutputParser()
)

# 6. Query
response = chain.invoke("Who is Abhishek Jaiswar ?")
print(response)

Abhishek Jaiswar is a Web Developer. He has 1 year of professional experience at TCS and 3 years of hands-on personal project development. He is skilled in React, Tailwind CSS, Redux, Node.js, Express.js, and REST APIs.


In [65]:
response = chain.invoke("what do you think about Abhishek Jaiswar ?")
print(response)

KeyboardInterrupt: 